<a href="https://colab.research.google.com/github/tzuyin-19/GenerativeAI-projects/blob/main/%E4%BF%9D%E5%81%A5%E9%A3%9F%E5%93%81%E6%AF%94%E5%83%B9Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 保健食品比價｜Tool Use Agent

---



### Agent構想

這次作業我選擇 Tool Use。一開始，我想把保健食品的服用提醒、降價通知、比價和成分評估都做在同一個系統裡，但討論後發現這樣會讓作業範圍太大，也不容易聚焦在Agent機制。因此我把主題縮小成「**保健食品價格搜尋與比較**」，希望使用者只需要輸入商品名稱，AI 就能理解問題並選擇適合的工具。

### 資料選擇

我最初是參考老師的範例，先用自己建立的價格資料庫來測試工具查詢與排序；雖然這樣比較穩定，但查到的價格都是預先寫好的，使用情境比較受限。後來我想挑戰真實商品搜尋，所以改成串接 **ECPRO 的公開商品 API**。我也測試過 SerpApi，但結果不是很理想，所以最後選擇目前這個方式。

In [ ]:
%pip install -q "aisuite[groq]==0.2.0" "groq==0.37.1" "gradio==6.30.0" requests

### 載入模組

In [ ]:
import os
import json
import re
from datetime import datetime, timezone
from getpass import getpass
from importlib.metadata import version

import requests
import aisuite as ai
import gradio as gr

MODEL = "groq:openai/gpt-oss-120b"
ECPRO_URL = "https://ecpro.tw/api/v1/products"
print({name: version(name) for name in ["aisuite", "groq", "gradio"]})

{'aisuite': '0.2.0', 'groq': '0.37.1', 'gradio': '6.30.0'}


## 第一個工具：搜尋真實商品價格

### 搜尋工具的設計

第一個工具它會接收商品關鍵字，向公開 API (ECPRO)發出查詢，然後把回傳資料整理成商品名稱、商家、價格和連結。因為 API 回傳的欄位不一定完全一致，我另外處理了不同 JSON 欄位名稱、金額字串轉換，以及沒有有效結果時的錯誤訊息。

In [ ]:
def _extract_rows(payload):
    if isinstance(payload, list):
        return payload
    if not isinstance(payload, dict):
        return None
    for key in ("products", "results", "items", "offers", "data"):
        val = payload.get(key)
        if isinstance(val, list):
            return val
        if isinstance(val, dict):
            for inner in ("products", "results", "items", "offers", "data"):
                if isinstance(val.get(inner), list):
                    return val[inner]
    return None


def _money(value):
    if isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value) if value > 0 else None
    if isinstance(value, str):
        clean = value.replace(",", "").replace("NT$", "").replace("NTD", "").replace("TWD", "").replace("$", "").strip()
        if re.fullmatch(r"\d+(?:\.\d+)?", clean):
            number = float(clean)
            return number if number > 0 else None
    return None


def _normalize_products(rows):
    normalized = []
    for row in rows[:60]:
        if not isinstance(row, dict):
            continue
        title = str(row.get("title") or row.get("name") or row.get("product_name") or "")
        vendor = row.get("store") or row.get("source") or row.get("merchant") or row.get("platform") or row.get("shop") or "未標示"
        price = _money(row.get("price") or row.get("sale_price") or row.get("current_price") or row.get("amount"))
        if not title or price is None:
            continue
        currency = str(row.get("currency") or "TWD（API未明示時須查核）")
        normalized.append({
            "title": title,
            "merchant": str(vendor),
            "price": round(price, 2),
            "currency": currency,
            "link": str(row.get("url") or row.get("link") or row.get("product_url") or ""),
        })
    return normalized


def search_prices(query):
    """Tool 1：從真實公開 API 搜尋商品；不猜價、不回傳虛構資料。"""
    if not isinstance(query, str) or not 2 <= len(query.strip()) <= 120:
        return {"ok": False, "error": "請提供 2～120 字的明確商品名稱。"}
    try:
        response = requests.get(
            ECPRO_URL,
            params={"q": query.strip()},
            headers={"Accept": "application/json", "User-Agent": "StudentToolUseAgent/1.0"},
            timeout=18,
        )
        response.raise_for_status()
        payload = response.json()
    except (requests.RequestException, ValueError) as exc:
        return {"ok": False, "source": "ECPRO", "error_type": type(exc).__name__,
                "error": "商品 API 連線失敗、回傳 HTTP 錯誤，或無法解讀 JSON。"}
    if isinstance(payload, dict) and payload.get("error"):
        return {"ok": False, "source": "ECPRO", "error": str(payload["error"])[:300]}
    rows = _extract_rows(payload)
    if rows is None:
        return {"ok": False, "source": "ECPRO",
                "error": "API 已回覆，但回傳格式不符合目前支援的商品清單結構。",
                "top_level_keys": list(payload.keys())[:15] if isinstance(payload, dict) else []}
    offers = _normalize_products(rows)
    return {"ok": bool(offers), "source": "ECPRO 電商博士", "query": query.strip(),
            "checked_at_utc": datetime.now(timezone.utc).isoformat(),
            "offers": offers[:15],
            "error": None if offers else "沒有取得含有效商品名稱與價格的結果。",
            "note": "真實索引資料，不保證所有商家完整涵蓋；請核對規格、幣別、運費與最終結帳價。"}

## 第二個工具：比較價格

第一次在 Gradio 測試「速攻藍莓」時，Agent A 確實選到了比價工具，API 也回傳了商品，但搜尋結果混入了不同配方、品牌或包裝規格。原本只按照售價排序，反而可能把不相同的商品當作同款比較。

我在檢查 Agent B 的回覆時發現，它也提出了相同疑問：**這些商品真的都是同一款、同一規格嗎？** 因此我修改了一些程式碼，先從標題初步檢查品牌、維他命 C 類型、日份及粒數／錠數，再對通過的結果排序。這不是完美的商品辨識，但至少能避免直接用不相關的低價商品得出結論；找不到相符品項時，也會明確回報。

In [ ]:
def check_product_match(query, title):
    """檢查搜尋結果是否符合指定品牌、成分及規格。"""

    def clean(text):
        return re.sub(r"\s+", "", text).lower()

    q = clean(query)
    t = clean(title)

    brands = ["dhc", "nowfoods", "善存", "萊萃美"]

    for brand in brands:
        if brand in q and brand not in t:
            return False
    if "維他命c" in q or "維生素c" in q:
        if "維他命c" not in t and "維生素c" not in t:
            return False
        if any(word in t for word in ["b群", "維他命b", "維生素b"]):
            if not any(word in q for word in ["b群", "維他命b", "維生素b"]):
                return False
    days = re.search(r"(\d+)日份", q)

    if days:
        target_days = days.group(1)
        if target_days + "日份" not in t:
            return False
    for number, unit in re.findall(r"(\d+)(粒|錠|顆)", q):
        if number + unit not in t:
            return False

    return True


def compare_prices(query):
    """查詢商品、篩選相符規格，再比較價格。"""

    result = search_prices(query)

    if not result["ok"]:
        return result
    matched = [
        item for item in result["offers"]
        if check_product_match(query, item["title"])
    ]

    if not matched:
        return {
            "ok": False,
            "query": query,
            "error": "沒有找到名稱及規格符合條件的商品。",
            "total_found": len(result["offers"]),
            "matched_count": 0,
            "note": "搜尋到的商品可能屬於不同規格。"
        }
    matched = sorted(
        matched,
        key=lambda item: item["price"]
    )

    return {
        **result,
        "offers": matched,
        "total_found": len(result["offers"]),
        "matched_count": len(matched),
        "lowest_observed_price": matched[0]["price"],
        "lowest_observed_merchant": matched[0]["merchant"],
        "comparison_warning": (
            "已進行商品名稱與規格初步篩選，"
            "但仍需人工確認完整成分、包裝數、"
            "幣別、運費及結帳價格。"
        )
    }

## 連接 Groq

### 模型與資料的分工

讓 LLM 負責理解使用者意圖、決定要呼叫哪個工具，以及用自然語言整理結果；相對地，查價、金額解析、篩選和排序交給 Python。這樣做是為了避免 AI 單憑語言生成能力猜測商品價格。

In [ ]:
api_key = ""
try:
    from google.colab import userdata
    api_key = userdata.get("Groq") or ""
except Exception:
    pass
if not api_key.strip():
    api_key = getpass("請輸入 Groq API Key（不會顯示）：")
if not api_key.strip():
    raise ValueError("尚未提供 Groq API Key。")
os.environ["GROQ_API_KEY"] = api_key.strip()
del api_key

client = ai.Client(provider_configs={"groq": {"timeout": 60.0, "max_retries": 0}})

###呼叫 LLM

In [ ]:
def call_llm(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_prompt}],
        temperature=0,
        max_tokens=1500,
    )
    text = response.choices[0].message.content
    if not isinstance(text, str) or not text.strip():
        raise RuntimeError("模型未回傳文字。")
    return text.strip()

## 第一階段：Agent A 選擇工具

### Agent A 的選擇

設計不同的使用情境：當使用者問「哪裡有賣」時選 search_price ；問「哪一家便宜」時選 compare_prices；如果連商品名稱都沒有提供，就改成 clarify 請使用者補充。

第一次 LLM 的任務只會產生 JSON 工具請求，還不會執行搜尋。這讓我能在 Gradio 裡直接觀察 AI 到底選了什麼工具、帶入什麼參數，而不是只看最後一段回答。

In [ ]:
ROUTING_PROMPT = """你是 AI 保健食品比價的工具路由器 Agent A。
只輸出單一合法 JSON 物件，不輸出 Markdown。
可用：
{"tool":"search_prices","arguments":{"query":"具體的商品關鍵字"}}
{"tool":"compare_prices","arguments":{"query":"具體的商品關鍵字與規格"}}
{"tool":"clarify","arguments":{"question":"請補充的商品資訊"}}
規則：
- 問有哪些賣家、有沒有商品、哪裡有賣：search_prices。
- 問哪個更便宜、比較報價、最低標價：compare_prices。
- 完全沒說商品名稱或關鍵字：clarify。
- 只擷取使用者指定的品牌、名稱、容量；不可自創規格或價格。
- 一次選一個工具，不做購買、不提供醫療建議。"""
question = "幫我比較 DHC 速攻藍莓 哪個商家價格比較便宜？"
request_text = call_llm(ROUTING_PROMPT, question)
print(request_text)

{"tool":"compare_prices","arguments":{"query":"DHC 速攻藍莓"}}


## 第二階段：Python 解析與執行工具

In [ ]:
def parse_request(text):
    raw = text.strip()
    if raw.startswith("```"):
        lines = raw.splitlines()
        if lines[0].strip().lower() not in ("```", "```json") or lines[-1].strip() != "```":
            raise ValueError("不正確的 JSON 程式框。")
        raw = "\n".join(lines[1:-1])
    try:
        req = json.loads(raw)
    except json.JSONDecodeError:
        raise ValueError("LLM 未回傳合法 JSON。") from None
    if not isinstance(req, dict) or set(req) != {"tool", "arguments"} or not isinstance(req["arguments"], dict):
        raise ValueError("工具請求格式錯誤。")
    return req


def execute_request(req):
    name, args = req["tool"], req["arguments"]
    if name in ("search_prices", "compare_prices"):
        if set(args) != {"query"} or not isinstance(args["query"], str):
            raise ValueError("查價工具只接受 query 字串。")
        return (search_prices if name == "search_prices" else compare_prices)(args["query"])
    if name == "clarify":
        if set(args) != {"question"} or not isinstance(args["question"], str) or not args["question"].strip():
            raise ValueError("clarify 必須有非空 question。")
        return {"needs_clarification": True, "question": args["question"]}
    raise ValueError("不允許的工具。")

### 執行第一次 LLM 選定的工具

In [ ]:
request = parse_request(request_text)
tool_result = execute_request(request)
print("工具請求：")
print(json.dumps(request, ensure_ascii=False, indent=2))
print("實際 Python 結果：")
print(json.dumps(tool_result, ensure_ascii=False, indent=2)[:8000])

工具請求：
{
  "tool": "compare_prices",
  "arguments": {
    "query": "DHC 速攻藍莓"
  }
}
實際 Python 結果：
{
  "ok": true,
  "source": "ECPRO 電商博士",
  "query": "DHC 速攻藍莓",
  "checked_at_utc": "2026-11-03T14:33:57.858811+00:00",
  "offers": [
    {
      "title": "【現貨馬上出】🔥 日本 DHC保健品 GABA 速攻藍莓 上班族必備 營養保健品  日本正版",
      "merchant": "未標示",
      "price": 99.0,
      "currency": "TWD（API未明示時須查核）",
      "link": "https://anything.com.tw/product/%e3%80%90%e7%8f%be%e8%b2%a8%e3%80%91%f0%9f%94%a5%e6%97%a5%e6%9c%ac-dhc-gaba-20%e6%97%a5%e8%a3%9d-%e6%83%85%e7%b7%92-%e5%a3%93%e5%8a%9b-%e7%87%9f%e9%a4%8a%e4%bf%9d%e5%81%a5%e5%93%81-%e6%ad%a3%e7%89%88/"
    },
    {
      "title": "DHC 速攻藍莓 三倍修護保護營養膠囊 緩解眼睛疲勞乾澀 30日分60粒 | 20日分40粒",
      "merchant": "未標示",
      "price": 193.0,
      "currency": "TWD（API未明示時須查核）",
      "link": "https://vvshop.com.tw/product/dhc-%e9%80%9f%e6%94%bb%e8%97%8d%e8%8e%93-%e4%b8%89%e5%80%8d%e4%bf%ae%e8%ad%b7%e4%bf%9d%e8%ad%b7%e7%87%9f%e9%a4%8a%e8%86%a0%e5%9b%8a-%e7%b7%a9%e8%a7%a3%e7%9c%b

##運作流程： Agent A 初步回答 → Agent B 審核 → Agent A 修正

### 為什麼加入 Agent B

完成單一 Agent 的查價流程後，我希望結果不只是「搜尋完就回答」，而是可以再多一道檢查。因此我把原本的回答階段擴充成兩個角色：**Agent A 是搜尋員，Agent B 是審核員**。Agent A 先依工具結果寫初稿；Agent B 讀取同一份工具資料，對商品規格、價格來源或比較方式提出問題；接著 Agent A 再根據審核意見修正回答。

In [ ]:
ANSWER_PROMPT = """你是商品搜尋員 Agent A，使用繁體中文簡潔回答。
只能引用 actual_result 中的實際價格與來源，不能編造報價或商品。
如果 API 失敗，請明確說明沒有取得有效資料，不准猜測。
有報價時最多列五項，表格中只包含：商品名稱、價格與連結（請完全去除並不要顯示商家欄位）；商品規格未驗證，不能宣稱完全同款或全網最便宜。
資料源是 ECPRO 電商博士。提醒使用者核對規格、幣別、運費和結帳價。
如果 review 提出疑問，請根據 actual_result 回答並修正，不得憑空補充。
不提供保健食品服用或醫療建議。"""
REVIEW_PROMPT = """你是商品比價審核員 Agent B，使用繁體中文。
閱讀原問題、搜尋員初稿及實際 Python 工具資料，向 Agent A 提出一項最重要的核查問題。
重點檢查是否編造價格、把不同規格誤認同款、誤稱全網最低價、忽略查詢失敗或運費。
用一至三句話提問即可。不能創造新的價格。"""

def make_answer_prompt(question, request, result, draft=None, review=None):
    pack = {"original_question": question, "request": request, "actual_result": result}
    if draft is not None:
        pack["previous_draft"] = draft
    if review is not None:
        pack["review"] = review
    return json.dumps(pack, ensure_ascii=False, indent=2)

answer_prompt = make_answer_prompt(question, request, tool_result)
draft = call_llm(ANSWER_PROMPT, answer_prompt)
review = call_llm(REVIEW_PROMPT, json.dumps({
    "original_question": question, "actual_result": tool_result, "draft": draft
}, ensure_ascii=False))
answer = call_llm(ANSWER_PROMPT, make_answer_prompt(
    question, request, tool_result, draft=draft, review=review))
print("Agent A 初稿：", draft)
print("Agent B 問題：", review)
print("Agent A 修正：", answer)

Agent A 初稿： 以下為 **ECPRO 電商博士** 取得的 DHC 速攻藍莓 相關報價（價格單位為 TWD，實際結帳金額請自行確認幣別、運費與稅金）：

| 商品名稱 | 價格 (TWD) | 連結 |
|---|---|---|
| 【現貨馬上出】🔥 日本 DHC保健品 GABA 速攻藍莓 上班族必備 營養保健品 日本正版 | 99 | https://anything.com.tw/product/%e3%80%90%e7%8f%be%e8%b2%a8%e3%80%91%f0%9f%94%a5%e6%97%a5%e6%9c%ac-dhc-gaba-20%e6%97%a5%e8%a3%9d-%e6%83%85%e7%b7%92-%e5%a3%93%e5%8a%9b-%e7%87%9f%e9%a4%8a%e4%bf%9d%e5%81%a5%e5%93%81-%e6%ad%a3%e7%89%88/ |
| DHC 速攻藍莓 三倍修護保護營養膠囊 緩解眼睛疲勞乾澀 30日分60粒 \| 20日分40粒 | 193 | https://vvshop.com.tw/product/dhc-%e9%80%9f%e6%94%bb%e8%97%8d%e8%8e%93-%e4%b8%89%e5%80%8d%e4%bf%ae%e8%ad%b7%e4%bf%9d%e8%ad%b7%e7%87%9f%e9%a4%8a%e8%86%a0%e5%9b%8a-%e7%b7%a9%e8%a7%a3%e7%9c%bc%e7%9d%9b%e7%96%b2%e5%8b%9e%e4%b9%be/ |
| DHC 速攻藍莓 V-MAX 史上最強的清晰支援補充品 30日分60粒 | 574 | https://vvshop.com.tw/product/dhc-%e9%80%9f%e6%94%bb%e8%97%8d%e8%8e%93-v-max-%e5%8f%b2%e4%b8%8a%e6%9c%80%e5%bc%b7%e7%9a%84%e6%b8%85%e6%99%b0%e6%94%af%e6%8f%b4%e8%a3%9c%e5%85%85%e5%93%81-30%e6%97%a5%e5%88%8660%e7%b2%92/ |
| 【DHC】速攻藍莓 30日份 3入組(60粒/包/花青素/葉黃素)

### 整合整個流程

為了方便 Gradio 使用，我把前面分開示範的流程整合。每次提問會依序產生工具請求、執行 Python 工具、生成 Agent A 初稿、取得 Agent B 的審核問題，再產生 Agent A 的修正答案。

In [ ]:
def run_once(question):
    trace = {"request_text": "", "request": None, "result": None,
             "draft": "", "review": "", "answer": "", "status": ""}
    if not isinstance(question, str) or not question.strip():
        trace["status"] = "請輸入商品名稱與問題。"
        return trace
    if len(question) > 300:
        trace["status"] = "輸入請控制在 300 字內。"
        return trace
    stage = "Agent A 選工具"
    try:
        trace["request_text"] = call_llm(ROUTING_PROMPT, question)
        trace["request"] = parse_request(trace["request_text"])
        stage = "Python 工具執行"
        trace["result"] = execute_request(trace["request"])
        stage = "Agent A 初稿"
        trace["draft"] = call_llm(ANSWER_PROMPT, make_answer_prompt(question, trace["request"], trace["result"]))
        stage = "Agent B 審核"
        trace["review"] = call_llm(REVIEW_PROMPT, json.dumps({
            "original_question": question,
            "actual_result": trace["result"],
            "draft": trace["draft"],
        }, ensure_ascii=False))
        stage = "Agent A 修正"
        trace["answer"] = call_llm(ANSWER_PROMPT, make_answer_prompt(
            question, trace["request"], trace["result"],
            draft=trace["draft"], review=trace["review"]))
        trace["status"] = "已完成工具選擇、實際查詢、雙角色審核與修正。"
    except Exception as exc:
        trace["status"] = f"{stage}失敗：{type(exc).__name__}。請檢查 Groq 額度、API 回應或 JSON 格式。"
    return trace

## 測試與改良

### 測試方式與我的觀察

測試分成不同類型：一般商品搜尋、指定商品比價、資訊不足，以及不合法的工具請求。一開始雖然有查到價格，但不能直接視為有效的同款比價；後來我才加入商品標題的初步篩選，讓結果更保守。

這也讓我發現，Agent 成功呼叫工具不代表最終答案一定正確。除了看工具有沒有執行，更需要核對工具查到的是不是使用者真正想找的商品。

In [ ]:
bad_requests = [
    {"tool": "delete_database", "arguments": {}},
    {"tool": "compare_prices", "arguments": {"query": 8}},
]
for bad in bad_requests:
    try:
        execute_request(bad)
    except ValueError as error:
        print("成功拒絕：", error)
    else:
        raise AssertionError("不允許的工具被執行")

成功拒絕： 不允許的工具。
成功拒絕： 查價工具只接受 query 字串。


## Gradio 展示完整流程

### Gradio 介面呈現的重點

Gradio會顯示 **Agent A 選擇的工具 JSON、Python 實際查詢結果、Agent A 初稿、Agent B 審核問題、Agent A 修正結果** 。這樣可以直接展示的運作過程，也能看見Agent B是否真的對初稿提出檢查。

In [ ]:
def gradio_answer(question):
    t = run_once(question)
    return t["answer"], t["draft"], t["review"], t["request_text"], t["request"], t["result"], t["status"]

if "demo" in globals():
    demo.close()

with gr.Blocks(title="Supplement Price Hunter") as demo:
    gr.Markdown("""
    # Supplement Price Hunter
    **AI 保健食品比價助手**

    輸入保健食品名稱，AI 將搜尋真實商品資料、比較價格，並透過兩個 Agent 角色審核結果。

    資料來源：ECPRO 電商博士公開 API
    """)

    with gr.Group():
        question_box = gr.Textbox(label="想比較哪款保健食品？", placeholder="例如：幫我比較 DHC 速攻藍莓 哪個商家比較便宜？", lines=2)
        run_button = gr.Button("開始搜尋與比價", variant="primary")

    gr.Examples(
        examples=[
            ["哪裡有賣 DHC 速攻藍莓？"],
            ["幫我比較 DHC 速攻藍莓 哪個商家比較便宜？"],
            ["幫我找最便宜的保健食品"]
        ],
        inputs=question_box,
        cache_examples=False
    )

    gr.Markdown("## 最終比價結果")
    answer_box = gr.Markdown(value="輸入商品名稱並按下開始比價，結果將顯示在這裡。")
    status_box = gr.Textbox(label="執行狀態", interactive=False)

    gr.Markdown("## AI Agent 協作過程")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Agent A｜商品搜尋員")
            draft_box = gr.Textbox(label="初步分析", lines=7, interactive=False)
        with gr.Column():
            gr.Markdown("### Agent B｜價格審核員")
            review_box = gr.Textbox(label="審核與提問", lines=7, interactive=False)

    gr.Markdown("Agent A 會根據 Agent B 的審核意見修正回答，最終結果顯示於上方。")

    with gr.Accordion("查看 Tool Use 執行紀錄", open=False):
        raw_box = gr.Textbox(label="Agent A 選擇工具的原始 JSON", lines=3, interactive=False)
        request_box = gr.JSON(label="Python 解析後的工具參數")
        result_box = gr.JSON(label="Python 實際查價結果")

    run_button.click(
        fn=gradio_answer,
        inputs=question_box,
        outputs=[answer_box, draft_box, review_box, raw_box, request_box, result_box, status_box]
    )

Closing server running on port: 7860


### 啟動 Gradio

In [ ]:
START_DEMO = True
if START_DEMO:
    demo.queue(max_size=8, default_concurrency_limit=1)
    demo.launch(share=True, inline=True, show_error=False)
else:
    print("確認單格查詢、Groq 呼叫都成功後，改成 START_DEMO = True。")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d99c09280c74679020.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 停止展示

In [ ]:
STOP_DEMO = False
if STOP_DEMO:
    demo.close()
    print("已停止展示。")